# Experiment: SNANA 首次探测时间差分布（BNS + NSBH）

目标：
- 使用原始 SNANA 模拟数据，统计首次探测与 GW 发生时刻的时间差 `delta_days = first_detect_mjd - MJD_EXPLODE`。
- 首次探测判定改为与 optical-only 数据集一致：先做同波段 2 小时内合并，再用 merged `psfFlux / psfFluxErr > 5` 定义首次探测。
- `MJD_EXPLODE` 从每个事件目录对应的 `README` 文件解析。
- 运行结束后直接覆盖 canonical offset 文件：`<BASE_DIR>/data/Optical_Only_dataset/delta_days_distribution.npz`。

数据源：
- `<BASE_DIR>/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS_AUG`
- `<BASE_DIR>/SNANA/SNDATA_ROOT/SIM/LSST_KN_NSBH_TRAIN`


In [ ]:
from __future__ import annotations

import importlib.util
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from tqdm.auto import tqdm

# ===== User Config =====
PROJECT_DIR = Path('<BASE_DIR>/gw-kn-multimodal/optical_only')
MODEL_DIR = Path('<BASE_DIR>/gw-kn-multimodal/Model')
OUTPUT_BASE_DIR = PROJECT_DIR / 'outputs' / 'snana_first_detection_delay_bns_nsbh'
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
CANONICAL_OFFSET_NPZ = Path('<BASE_DIR>/data/Optical_Only_dataset/delta_days_distribution.npz')

SIM_SOURCES = [
    {
        'source': 'BNS',
        'base_dir': Path('<BASE_DIR>/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS_AUG'),
        'prefix': 'LSST_KN_BNS_AUG',
    },
    {
        'source': 'NSBH',
        'base_dir': Path('<BASE_DIR>/SNANA/SNDATA_ROOT/SIM/LSST_KN_NSBH_TRAIN'),
        'prefix': 'LSST_KN_NSBH_TRAIN',
    },
]

SNR_THRESHOLD = 5.0
FLUXCAL_ZP = 27.5
PSFFLUX_ZP = 31.4
FLUXCAL_TO_PSFFLUX_FACTOR = 10.0 ** (0.4 * (PSFFLUX_ZP - FLUXCAL_ZP))

_MERGE_HELPER_PATH = MODEL_DIR / 'lightcurve_merge.py'
_merge_spec = importlib.util.spec_from_file_location('lightcurve_merge', _MERGE_HELPER_PATH)
if _merge_spec is None or _merge_spec.loader is None:
    raise ImportError(f'Unable to load merge helper from {_MERGE_HELPER_PATH}')
_lightcurve_merge = importlib.util.module_from_spec(_merge_spec)
_merge_spec.loader.exec_module(_lightcurve_merge)
MERGE_WINDOW_HOURS = float(_lightcurve_merge.MERGE_WINDOW_HOURS)
merge_photometry_psfflux = _lightcurve_merge.merge_photometry_psfflux

# 如果先做小样本验证，把 RUN_FULL_SCAN 设为 False。
RUN_FULL_SCAN = True
MAX_EVENTS_PER_SOURCE = None if RUN_FULL_SCAN else 20

USE_PARALLEL = False
N_WORKERS = min(8, max(1, (os.cpu_count() or 1) // 2))

# 是否额外保存逐 realization 的详细 delta CSV（可能很大）
SAVE_DETAILED_DELTAS_CSV = False

print('RUN_FULL_SCAN =', RUN_FULL_SCAN)
print('MAX_EVENTS_PER_SOURCE =', MAX_EVENTS_PER_SOURCE)
print('USE_PARALLEL =', USE_PARALLEL, 'N_WORKERS =', N_WORKERS)
print('SNR_THRESHOLD =', SNR_THRESHOLD)
print('MERGE_WINDOW_HOURS =', MERGE_WINDOW_HOURS)
print('FLUXCAL_TO_PSFFLUX_FACTOR =', FLUXCAL_TO_PSFFLUX_FACTOR)
print('CANONICAL_OFFSET_NPZ =', CANONICAL_OFFSET_NPZ)


## 方法说明

对每个事件目录（例如 `LSST_KN_BNS_AUG_123`）：
1. 从 `*.README` 解析 `MJD_EXPLODE`。
2. 从 `*_HEAD.FITS` 读取每条 realization 的 `PTROBS_MIN/PTROBS_MAX`，定位其在 `*_PHOT.FITS` 中的观测区间。
3. 将该区间里的观测按与 `create_optical_only_datasets.py` 相同的规则做同波段 2 小时内合并，并把 `FLUXCAL/FLUXCALERR` 转成 merged `psfFlux/psfFluxErr`。
4. 在 merged 序列中找第一个满足 `psfFlux / psfFluxErr > 5` 的观测点，记其 `MJD` 为 `first_detect_mjd`。
5. 计算 `delta_days = first_detect_mjd - MJD_EXPLODE`。

最终输出：
- BNS、NSBH 以及合并（Combined）的分布统计。
- 分布数组（`npz`）、统计表（`csv`）和直方图（`png`）。
- 同步覆盖 canonical offset 文件，供 optical-only 训练/评估/Fink 推理直接使用。


In [ ]:
MJD_EXPLODE_PATTERN = re.compile(r"MJD_EXPLODE:\s*([+-]?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)")


def parse_mjd_explode_from_readme(readme_path: Path) -> float:
    text = readme_path.read_text(encoding='utf-8', errors='ignore')
    m = MJD_EXPLODE_PATTERN.search(text)
    if m is None:
        raise ValueError(f'MJD_EXPLODE not found in README: {readme_path}')
    return float(m.group(1))


def build_detection_mask(flux: np.ndarray, fluxerr: np.ndarray, snr_threshold: float) -> np.ndarray:
    if flux.size == 0:
        return np.zeros((0,), dtype=bool)
    valid = np.isfinite(flux) & np.isfinite(fluxerr) & (fluxerr > 0)
    if not np.any(valid):
        return np.zeros(flux.shape, dtype=bool)
    snr = np.full(flux.shape, -np.inf, dtype=np.float64)
    snr[valid] = flux[valid] / fluxerr[valid]
    return np.asarray(snr > float(snr_threshold), dtype=bool)


def process_event_dir(
    event_dir: Path,
    source: str,
    snr_threshold: float,
    fluxcal_to_psfflux_factor: float,
) -> Dict:
    prefix = event_dir.name
    readme_path = event_dir / f'{prefix}.README'
    head_path = event_dir / f'{prefix}_HEAD.FITS'
    phot_path = event_dir / f'{prefix}_PHOT.FITS'

    event_id = -1
    try:
        event_id = int(prefix.rsplit('_', 1)[-1])
    except Exception:
        pass

    result = {
        'source': source,
        'event_dir': prefix,
        'event_id': event_id,
        'n_realizations': 0,
        'n_detected': 0,
        'n_no_detect': 0,
        'delta_days': np.empty((0,), dtype=np.float32),
        'error': None,
    }

    if not (readme_path.exists() and head_path.exists() and phot_path.exists()):
        result['error'] = 'missing_required_files'
        return result

    try:
        mjd_explode = parse_mjd_explode_from_readme(readme_path)

        with fits.open(head_path, memmap=False) as hdul_head, fits.open(phot_path, memmap=False) as hdul_phot:
            head = hdul_head[1].data
            phot = hdul_phot[1].data

            ptrobs_min = np.asarray(head['PTROBS_MIN'], dtype=np.int64)
            ptrobs_max = np.asarray(head['PTROBS_MAX'], dtype=np.int64)
            mjd_all = np.asarray(phot['MJD'], dtype=np.float64)
            flux_all = np.asarray(phot['FLUXCAL'], dtype=np.float64)
            fluxerr_all = np.asarray(phot['FLUXCALERR'], dtype=np.float64)
            flt_all = np.asarray(phot['BAND'])

        n_realizations = int(len(ptrobs_min))
        if n_realizations == 0:
            result['n_realizations'] = 0
            result['n_no_detect'] = 0
            return result

        deltas = np.empty((n_realizations,), dtype=np.float32)
        detected_mask = np.zeros((n_realizations,), dtype=bool)

        for i, (start_1based, end_1based) in enumerate(zip(ptrobs_min, ptrobs_max)):
            start = int(start_1based) - 1
            end = int(end_1based)

            if start < 0 or end <= start or end > mjd_all.size:
                continue

            local_mjd = mjd_all[start:end]
            local_flux = flux_all[start:end]
            local_fluxerr = fluxerr_all[start:end]
            local_flt = flt_all[start:end]

            merged_mjd, merged_psfflux, merged_psffluxerr, _ = merge_photometry_psfflux(
                mjd=local_mjd,
                fluxcal=local_flux,
                fluxcalerr=local_fluxerr,
                flt=local_flt,
                fluxcal_to_psfflux_factor=float(fluxcal_to_psfflux_factor),
            )
            if merged_mjd.size == 0:
                continue

            local_detect = build_detection_mask(merged_psfflux, merged_psffluxerr, snr_threshold=snr_threshold)
            if np.any(local_detect):
                first_local_idx = int(np.argmax(local_detect))
                first_detect_mjd = float(merged_mjd[first_local_idx])
                deltas[i] = np.float32(first_detect_mjd - mjd_explode)
                detected_mask[i] = True

        detected_deltas = deltas[detected_mask]
        n_detected = int(detected_deltas.size)

        result['n_realizations'] = n_realizations
        result['n_detected'] = n_detected
        result['n_no_detect'] = n_realizations - n_detected
        result['delta_days'] = detected_deltas
        return result

    except Exception as e:
        result['error'] = f'{type(e).__name__}: {e}'
        return result


def process_event_dir_worker(args: Tuple[str, str, float, float]) -> Dict:
    event_dir_str, source, snr_threshold, fluxcal_to_psfflux_factor = args
    return process_event_dir(
        Path(event_dir_str),
        source,
        float(snr_threshold),
        float(fluxcal_to_psfflux_factor),
    )


def list_event_dirs(base_dir: Path, prefix: str) -> List[Path]:
    return sorted([p for p in base_dir.glob(f'{prefix}_*') if p.is_dir()])


In [ ]:
def scan_source(
    source: str,
    base_dir: Path,
    prefix: str,
    snr_threshold: float,
    fluxcal_to_psfflux_factor: float,
    max_events: int | None = None,
    use_parallel: bool = False,
    n_workers: int = 1,
) -> Tuple[np.ndarray, pd.DataFrame, pd.DataFrame]:
    event_dirs = list_event_dirs(base_dir, prefix)
    if max_events is not None:
        event_dirs = event_dirs[: int(max_events)]

    print(f'[{source}] total event dirs selected: {len(event_dirs)}')

    delta_chunks: List[np.ndarray] = []
    event_rows: List[Dict] = []
    error_rows: List[Dict] = []

    def append_result(res: Dict) -> None:
        event_rows.append({
            'source': res['source'],
            'event_dir': res['event_dir'],
            'event_id': res['event_id'],
            'n_realizations': res['n_realizations'],
            'n_detected': res['n_detected'],
            'n_no_detect': res['n_no_detect'],
        })
        if res['delta_days'].size > 0:
            delta_chunks.append(res['delta_days'])
        if res['error'] is not None:
            error_rows.append({
                'source': res['source'],
                'event_dir': res['event_dir'],
                'event_id': res['event_id'],
                'error': res['error'],
            })

    ran_in_parallel = False
    if use_parallel and n_workers > 1 and len(event_dirs) > 0:
        args_iter = [
            (str(p), source, float(snr_threshold), float(fluxcal_to_psfflux_factor))
            for p in event_dirs
        ]
        try:
            with ProcessPoolExecutor(max_workers=n_workers) as ex:
                for res in tqdm(ex.map(process_event_dir_worker, args_iter, chunksize=16), total=len(args_iter), desc=f'{source} scan (parallel)'):
                    append_result(res)
            ran_in_parallel = True
        except Exception as e:
            print(f'[{source}] parallel mode failed ({type(e).__name__}: {e}); fallback to sequential mode.')
            delta_chunks.clear()
            event_rows.clear()
            error_rows.clear()

    if not ran_in_parallel:
        for event_dir in tqdm(event_dirs, desc=f'{source} scan (sequential)'):
            res = process_event_dir(
                event_dir,
                source,
                snr_threshold=snr_threshold,
                fluxcal_to_psfflux_factor=fluxcal_to_psfflux_factor,
            )
            append_result(res)

    delta_days = np.concatenate(delta_chunks).astype(np.float32) if delta_chunks else np.empty((0,), dtype=np.float32)
    event_df = pd.DataFrame(event_rows)
    error_df = pd.DataFrame(error_rows)

    return delta_days, event_df, error_df


all_deltas: Dict[str, np.ndarray] = {}
all_event_df: List[pd.DataFrame] = []
all_error_df: List[pd.DataFrame] = []

for cfg in SIM_SOURCES:
    source = cfg['source']
    delta_days, event_df, error_df = scan_source(
        source=source,
        base_dir=cfg['base_dir'],
        prefix=cfg['prefix'],
        snr_threshold=SNR_THRESHOLD,
        fluxcal_to_psfflux_factor=FLUXCAL_TO_PSFFLUX_FACTOR,
        max_events=MAX_EVENTS_PER_SOURCE,
        use_parallel=USE_PARALLEL,
        n_workers=N_WORKERS,
    )
    all_deltas[source] = delta_days
    all_event_df.append(event_df)
    if not error_df.empty:
        all_error_df.append(error_df)

bns_deltas = all_deltas.get('BNS', np.empty((0,), dtype=np.float32))
nsbh_deltas = all_deltas.get('NSBH', np.empty((0,), dtype=np.float32))
combined_deltas = np.concatenate([arr for arr in [bns_deltas, nsbh_deltas] if arr.size > 0]).astype(np.float32) if (bns_deltas.size + nsbh_deltas.size) > 0 else np.empty((0,), dtype=np.float32)

event_summary_df = pd.concat(all_event_df, ignore_index=True) if all_event_df else pd.DataFrame()
error_df = pd.concat(all_error_df, ignore_index=True) if all_error_df else pd.DataFrame(columns=['source', 'event_dir', 'event_id', 'error'])

print('BNS detected realizations:', bns_deltas.size)
print('NSBH detected realizations:', nsbh_deltas.size)
print('Combined detected realizations:', combined_deltas.size)
print('Events with processing errors:', len(error_df))


In [ ]:
def summarize_distribution(delta_days: np.ndarray, n_total_realizations: int, n_detected: int, source: str) -> Dict:
    out = {
        'source': source,
        'n_total_realizations': int(n_total_realizations),
        'n_detected': int(n_detected),
        'n_no_detect': int(n_total_realizations - n_detected),
        'detect_rate': float(n_detected / n_total_realizations) if n_total_realizations > 0 else np.nan,
        'mean_days': np.nan,
        'std_days': np.nan,
        'q01_days': np.nan,
        'q05_days': np.nan,
        'q25_days': np.nan,
        'q50_days': np.nan,
        'q75_days': np.nan,
        'q95_days': np.nan,
        'q99_days': np.nan,
        'min_days': np.nan,
        'max_days': np.nan,
    }

    if delta_days.size == 0:
        return out

    q = np.quantile(delta_days, [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    out.update({
        'mean_days': float(np.mean(delta_days)),
        'std_days': float(np.std(delta_days)),
        'q01_days': float(q[0]),
        'q05_days': float(q[1]),
        'q25_days': float(q[2]),
        'q50_days': float(q[3]),
        'q75_days': float(q[4]),
        'q95_days': float(q[5]),
        'q99_days': float(q[6]),
        'min_days': float(np.min(delta_days)),
        'max_days': float(np.max(delta_days)),
    })
    return out


if event_summary_df.empty:
    per_source_counts = pd.DataFrame(columns=['source', 'n_total_realizations', 'n_detected'])
else:
    per_source_counts = (
        event_summary_df
        .groupby('source', as_index=False)[['n_realizations', 'n_detected']]
        .sum()
        .rename(columns={'n_realizations': 'n_total_realizations'})
    )

summary_rows = []
for src_name, src_deltas in [('BNS', bns_deltas), ('NSBH', nsbh_deltas)]:
    row = per_source_counts.loc[per_source_counts['source'] == src_name]
    n_total = int(row['n_total_realizations'].iloc[0]) if len(row) else 0
    n_det = int(row['n_detected'].iloc[0]) if len(row) else 0
    summary_rows.append(summarize_distribution(src_deltas, n_total, n_det, src_name))

n_total_combined = int(per_source_counts['n_total_realizations'].sum()) if len(per_source_counts) else 0
n_detected_combined = int(per_source_counts['n_detected'].sum()) if len(per_source_counts) else 0
summary_rows.append(summarize_distribution(combined_deltas, n_total_combined, n_detected_combined, 'Combined'))

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
run_tag = pd.Timestamp.utcnow().strftime('%Y%m%dT%H%M%SZ')
run_dir = OUTPUT_BASE_DIR / run_tag
run_dir.mkdir(parents=True, exist_ok=True)

# 1) Save summary tables
summary_csv = run_dir / 'summary_stats.csv'
summary_df.to_csv(summary_csv, index=False)

event_csv = run_dir / 'event_level_counts.csv'
event_summary_df.to_csv(event_csv, index=False)

error_csv = run_dir / 'event_errors.csv'
error_df.to_csv(error_csv, index=False)

# 2) Save distribution arrays
npz_path = run_dir / 'delta_days_distribution.npz'
np.savez_compressed(
    npz_path,
    delta_days_bns=bns_deltas,
    delta_days_nsbh=nsbh_deltas,
    delta_days_combined=combined_deltas,
    snr_detection_threshold=np.float32(SNR_THRESHOLD),
    merge_window_hours=np.float32(MERGE_WINDOW_HOURS),
    fluxcal_to_psfflux_factor=np.float32(FLUXCAL_TO_PSFFLUX_FACTOR),
)

# Overwrite canonical offset file used by optical-only train/eval/inference
CANONICAL_OFFSET_NPZ.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    CANONICAL_OFFSET_NPZ,
    delta_days_bns=bns_deltas,
    delta_days_nsbh=nsbh_deltas,
    delta_days_combined=combined_deltas,
    snr_detection_threshold=np.float32(SNR_THRESHOLD),
    merge_window_hours=np.float32(MERGE_WINDOW_HOURS),
    fluxcal_to_psfflux_factor=np.float32(FLUXCAL_TO_PSFFLUX_FACTOR),
)

# Optional: save detailed per-realization deltas as CSV (can be very large)
if SAVE_DETAILED_DELTAS_CSV:
    rows = []
    if bns_deltas.size > 0:
        rows.append(pd.DataFrame({'source': 'BNS', 'delta_days': bns_deltas}))
    if nsbh_deltas.size > 0:
        rows.append(pd.DataFrame({'source': 'NSBH', 'delta_days': nsbh_deltas}))
    if rows:
        pd.concat(rows, ignore_index=True).to_csv(run_dir / 'detected_delta_days_detailed.csv', index=False)

# 3) Histogram + plot
if combined_deltas.size > 0:
    q_lo = float(np.quantile(combined_deltas, 0.001))
    q_hi = float(np.quantile(combined_deltas, 0.999))
    lo = np.floor(min(q_lo, 0.0))
    hi = np.ceil(max(q_hi, 10.0))
    if hi <= lo:
        hi = lo + 1.0

    bins = np.linspace(lo, hi, 300)

    hist_bns, edges = np.histogram(bns_deltas, bins=bins) if bns_deltas.size > 0 else (np.zeros(len(bins)-1, dtype=int), bins)
    hist_nsbh, _ = np.histogram(nsbh_deltas, bins=bins) if nsbh_deltas.size > 0 else (np.zeros(len(bins)-1, dtype=int), bins)
    hist_combined, _ = np.histogram(combined_deltas, bins=bins)

    hist_df = pd.DataFrame({
        'bin_left': edges[:-1],
        'bin_right': edges[1:],
        'count_bns': hist_bns,
        'count_nsbh': hist_nsbh,
        'count_combined': hist_combined,
    })
    hist_df.to_csv(run_dir / 'histogram_counts.csv', index=False)

    plt.figure(figsize=(10, 5))
    plt.step(edges[:-1], hist_combined, where='post', label='Combined', linewidth=2)
    if bns_deltas.size > 0:
        plt.step(edges[:-1], hist_bns, where='post', label='BNS', alpha=0.9)
    if nsbh_deltas.size > 0:
        plt.step(edges[:-1], hist_nsbh, where='post', label='NSBH', alpha=0.9)

    plt.axvline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.7)
    plt.xlabel('Merged-SNR first detection delay (days) = first_detect_mjd - MJD_EXPLODE')
    plt.ylabel('Count')
    plt.title('SNANA first-detection delay distribution (merged same-band 2h + psfFlux SNR > 5)')
    plt.legend()
    plt.tight_layout()
    plot_path = run_dir / 'first_detection_delay_histogram.png'
    plt.savefig(plot_path, dpi=180)
    plt.show()
else:
    print('No detected realizations found; skip histogram plotting.')

# 4) Save latest run pointer
(OUTPUT_BASE_DIR / 'latest_run.txt').write_text(str(run_dir), encoding='utf-8')

print('Saved outputs to:', run_dir)
print('summary_csv:', summary_csv)
print('event_csv:', event_csv)
print('error_csv:', error_csv)
print('npz_path:', npz_path)
print('canonical_npz_path:', CANONICAL_OFFSET_NPZ)


## 使用建议

- 小样本验证：`RUN_FULL_SCAN=False`，先跑通逻辑与输出格式。
- 全量统计：把 `RUN_FULL_SCAN=True`（或把 `MAX_EVENTS_PER_SOURCE=None`），建议在计算节点运行。
- 若需要保存每条 realization 的明细分布，设置 `SAVE_DETAILED_DELTAS_CSV=True`（文件可能很大）。
